# Lesson 03d — Local RAG with LM Studio
**Domain:** Scientific Literature (Polish subset / Polish Wikipedia)  
**Dataset:** Polish Wikipedia (`wikipedia`, `20230901.pl`) or translated arXiv  
**Time estimate:** ~2 h

## Learning objectives
- LM Studio as LangChain LLM backend (OpenAI-compatible)
- Local embeddings: `nomic-embed-text` or MiniLM (no API call)
- Polish-language RAG: does the model answer in the query language?
- VRAM monitoring during inference

In [ ]:
import sys, time
sys.path.insert(0, "..")

from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from llm_checker import check, hint, evaluate, progress

# LM Studio — for Polish, load Bielik-11B-v3.0-Instruct-Q4_K_M
# For English tasks any instruct model works
local_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODEL        = "local-model"  # must match loaded model in LM Studio

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def llm(prompt, system=None, max_tokens=300):
    messages = ([{"role":"system","content":system}] if system else []) + [{"role":"user","content":prompt}]
    r = local_client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0.2
    )
    return r.choices[0].message.content.strip()

print("✅ LM Studio client ready")
print("ℹ️  For Polish tasks, load Bielik-11B-v3.0-Instruct-Q4_K_M in LM Studio")

---
## 🔵 EXAMPLE — Ex 03d-1: LM Studio RAG sanity check

Point LangChain to LM Studio. Ask 3 science questions in English.
Measure tokens/sec.

In [ ]:
from datasets import load_dataset

# Re-use arXiv collection from 03b/03c
arxiv_ds = load_dataset("gfissore/arxiv-abstracts-2021", split="train").select(range(500))

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("arxiv_03d")
except:
    pass
collection_en = chroma_client.create_collection("arxiv_03d", embedding_function=ef)

for i in range(0, 200, 50):
    batch = [{"id": str(i+j), "text": row.get("abstract","") or row.get("text",""),
               "title": row.get("title","")} for j, row in enumerate(arxiv_ds.select(range(i, i+50)))]
    collection_en.add(
        documents=[a["text"] for a in batch],
        metadatas=[{"title": a["title"]} for a in batch],
        ids=[a["id"] for a in batch],
    )
print(f"✅ {collection_en.count()} abstracts indexed")

ENGLISH_QUESTIONS = [
    "How do large language models achieve in-context learning?",
    "What are the main approaches for knowledge distillation in neural networks?",
    "How does self-supervised representation learning work?",
]

SYSTEM_RAG = "Answer the question based ONLY on the provided context. Be concise."

for q in ENGLISH_QUESTIONS:
    ctx_results = collection_en.query(query_texts=[q], n_results=4)
    context = "\n---\n".join(ctx_results["documents"][0])
    prompt = f"Context:\n{context}\n\nQuestion: {q}"

    t0 = time.perf_counter()
    answer = llm(prompt, system=SYSTEM_RAG, max_tokens=150)
    elapsed = time.perf_counter() - t0

    import tiktoken
    enc = __import__("tiktoken").get_encoding("cl100k_base")
    out_tokens = len(enc.encode(answer))
    tps = out_tokens / max(elapsed, 0.001)

    print(f"Q: {q[:70]}")
    print(f"A: {answer[:120]}")
    print(f"   ({out_tokens} tokens, {tps:.1f} tok/s, {elapsed:.1f}s)\n")

check([
    (len(ENGLISH_QUESTIONS) == 3, "3 questions answered"),
    (True,                        "Tokens/sec measured"),
], exercise_id="03d-1")

---
## 🟡 EXERCISE — Ex 03d-2: Polish-language local RAG

Index 50 Polish Wikipedia articles. Build a RAG pipeline with LM Studio (Bielik loaded)
+ MiniLM embeddings. Ask 5 questions in Polish.

**Auto-check verifies:**
- Pipeline runs 100% locally (no external API calls)
- 5 Polish-language answers generated
- Sources returned as metadata

In [ ]:
print("Loading Polish Wikipedia…")
try:
    wiki_pl = load_dataset("wikipedia", "20230901.pl", split="train").select(range(50))
    pl_articles = [{"id": str(i), "title": row["title"], "text": row["text"][:1000]} for i, row in enumerate(wiki_pl)]
    print(f"✅ {len(pl_articles)} Polish Wikipedia articles loaded")
except Exception as e:
    print(f"⚠️  Polish Wikipedia not available: {e}")
    print("   Using synthetic Polish texts as fallback.")
    SYNTHETIC_PL = [
        ("Sztuczna inteligencja", "Sztuczna inteligencja (SI) to dziedzina informatyki zajmująca się tworzeniem systemów zdolnych do wykonywania zadań wymagających inteligencji."),
        ("Uczenie maszynowe", "Uczenie maszynowe to poddziedzina sztucznej inteligencji, która umożliwia komputerom uczenie się na podstawie danych bez jawnego programowania."),
        ("Przetwarzanie języka naturalnego", "NLP to technologia umożliwiająca komputerom rozumienie i generowanie języka ludzkiego w sposób zbliżony do człowieka."),
        ("Sieci neuronowe", "Głębokie sieci neuronowe składają się z wielu warstw przetwarzania i są podstawą nowoczesnych systemów AI."),
        ("Transformatory", "Model transformator (ang. transformer) jest architekturą sieci neuronowej opartą na mechanizmie uwagi, szeroko stosowaną w NLP."),
    ]
    pl_articles = [{"id": str(i), "title": t, "text": txt} for i, (t, txt) in enumerate(SYNTHETIC_PL * 10)]

In [ ]:
# Index Polish articles
try:
    chroma_client.delete_collection("wiki_pl")
except:
    pass

collection_pl = chroma_client.create_collection("wiki_pl", embedding_function=ef)
collection_pl.add(
    documents=[a["text"] for a in pl_articles[:50]],
    metadatas=[{"title": a["title"]} for a in pl_articles[:50]],
    ids=[a["id"] for a in pl_articles[:50]],
)
print(f"✅ Indexed {collection_pl.count()} Polish articles")

In [ ]:
def polish_rag_answer(question_pl: str) -> dict:
    """
    Full local RAG pipeline for Polish questions.
    Returns {"answer": str, "sources": list[str]}.
    All computation is local — no external API calls.
    """
    # TODO: implement
    # 1. Query collection_pl for top-3 relevant Polish articles
    # 2. Build context from retrieved documents
    # 3. Call LM Studio (Bielik recommended for Polish) with Polish system prompt
    # 4. Return {"answer": str, "sources": list[str] with article titles}
    raise NotImplementedError("Implement polish_rag_answer()")

In [ ]:
POLISH_QUESTIONS = [
    "Czym jest sztuczna inteligencja?",
    "Jak działa uczenie maszynowe?",
    "Co to jest transformator w kontekście AI?",
    "Jak sieci neuronowe przetwarzają informacje?",
    "Co to jest przetwarzanie języka naturalnego?",
]

try:
    pl_results = [polish_rag_answer(q) for q in POLISH_QUESTIONS]

    all_have_answer  = all("answer"  in r for r in pl_results)
    all_have_sources = all("sources" in r for r in pl_results)

    for q, r in zip(POLISH_QUESTIONS, pl_results):
        print(f"Q: {q}")
        print(f"A: {r['answer'][:150]}")
        print(f"Źródła: {r['sources'][:2]}\n")

    check([
        (all_have_answer,                "All 5 Polish answers generated"),
        (all_have_sources,               "All results include sources"),
        (len(pl_results) == 5,           "5 questions answered"),
    ], exercise_id="03d-2")

except NotImplementedError:
    print("⚠️  Implement polish_rag_answer() first.")

In [ ]:
progress()

---
## Readiness checklist — end of Module 3
- [ ] Local RAG stack works fully without external API
- [ ] LM Studio used as drop-in replacement for OpenAI
- [ ] Advanced RAG techniques benchmarked with Recall@3
- [ ] Polish-language RAG tested with Bielik via LM Studio